In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from google.colab import drive

# 1. SETUP AMBIENTE E PERCORSI
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/WiFi-HAR/doppler_traces'
DRIVE_WEIGHTS_DIR = '/content/drive/MyDrive/WiFi-HAR/weights'
os.makedirs(DRIVE_WEIGHTS_DIR, exist_ok=True)

# Clona la repository
REPO_URL = 'https://github.com/teoosera/WiFi-HAR.git'
!rm -rf /content/repo_code 
!git clone {REPO_URL} /content/repo_code

# --- CORREZIONE DEL PATH ---
# Ora aggiungiamo al path la sottocartella specifica che Git ha creato
sys.path.append('/content/repo_code/WiFi-HAR') 

# Debug: Verifichiamo che i file siano lì
print("File trovati nella cartella:", os.listdir('/content/repo_code/WiFi-HAR'))

# Aggiunge la cartella al path per permettere l'importazione dei moduli .py
sys.path.append('/content/repo_code')

# ---------------------------------------------------------
# 2. IMPORTAZIONE DAI TUOI MODULI
# ---------------------------------------------------------
# Se vedi sottolineature gialle nell'editor di Colab, ignorale.
# Il codice funzionerà correttamente dopo il git clone.
from dataset import get_dataloaders
from models import CustomSimpleCNN, CustomResNet, CNN_GRU
from train import train_one_epoch
from evaluate import evaluate_decision_fusion

# ---------------------------------------------------------
# 3. INIZIALIZZAZIONE DEL MODELLO
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizzando device: {device}")

# Modelli disponibili (decommenta quello che vuoi usare):
# model = CustomSimpleCNN(num_classes=5).to(device)
model = CustomResNet(num_classes=5).to(device)
# model = CNN_GRU(num_classes=5).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# 4. CARICAMENTO DATI E TRAINING LOOP
# ---------------------------------------------------------
print(f"\nCerco i dati in: {DRIVE_DATA_DIR}")
train_loader, test_loader = get_dataloaders(DRIVE_DATA_DIR, batch_size=32)

num_epochs = 10
for epoch in range(num_epochs):
    print(f"\n--- Epoca {epoch+1}/{num_epochs} ---")
    
    # Addestramento
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    
    # Test con Decision Fusion
    evaluate_decision_fusion(model, test_loader, device)

# ---------------------------------------------------------
# 5. SALVATAGGIO DEI PESI SU GOOGLE DRIVE
# ---------------------------------------------------------
save_path = f"{DRIVE_WEIGHTS_DIR}/custom_resnet_finale.pth"
torch.save(model.state_dict(), save_path)
print(f"\nAddestramento concluso! Pesi salvati in: {save_path}")